### Housing

This notebook performs transfer leanring on two Housing datasets, where similar columns are used!

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import random
from core import LADTransferTreeBoost, LSTransferTreeBoost, MTransferTreeBoost
from utils import *
from baselines import *
import xgboost as xgb
from sklearn.model_selection import train_test_split
import itertools

In [15]:
seed_list = [1]
splitting_variable_list = ['median_income']

In [16]:
data = pd.read_csv('../datasets/california-housing.csv')
data.columns

data = data.dropna()
for col in data.select_dtypes(include=['object']).columns:
    data[col] = data[col].astype('category').cat.codes
data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,3
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,3
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,3
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,3
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,3
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,1
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,1
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,1
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,1


In [17]:
#print correlations with target
target_column = 'median_house_value'
correlations = data.corr()[target_column].drop(target_column)
print(correlations)


longitude            -0.045398
latitude             -0.144638
housing_median_age    0.106432
total_rooms           0.133294
total_bedrooms        0.049686
population           -0.025300
households            0.064894
median_income         0.688355
ocean_proximity       0.080488
Name: median_house_value, dtype: float64


In [18]:
#ablation study for LSTransferTreeBoost with Gaussian errors, with gaussian source domain errors
ablation_transfer_housing = pd.DataFrame(columns = ['seed', 'splitting_variable', 'method', 'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 
                                                          'val_rmse', 'val_mae', 'rmse', 'mae'])
v_list = [0.05, 0.1]
source_tree_size_list = [1,2]
target_tree_size_list = [1,2]
k_list = [0.01, 0.05]
m_0_list = [0.5, 0.9]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    source_tree_size_list,
    target_tree_size_list,
    k_list,
    m_0_list
))

for seed in seed_list:
    for splitting_variable in splitting_variable_list:
    
        #order based on this variable
        data_ = data.sort_values(by=splitting_variable)
        #remove this variable now
        data_ = data_.drop(columns = splitting_variable)
        #divide into source and target data
        data_source = data_[0:int(len(data_) / 4)]
        data_target = data_[int(3*len(data_) / 4):]

        predictor_columns = [col for col in data_.columns if col != target_column]

        #Split target into train, val, test

        data_target_train, data_target_temp = train_test_split(data_target, test_size = 0.98, random_state=seed)
        data_target_val, data_target_test = train_test_split(data_target_temp, test_size = 0.5, random_state=seed)
        print(len(data_target_train), len(data_target_val), len(data_target_test))

        X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

        X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
        X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
        X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])
        for config in param_grid:
            v, source_tree_size, target_tree_size, k, m_0 = config


            #Test for all methods!!!!

            method = f'LSTransferTreeBoost'
            fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                                    target_tree_size=target_tree_size, k=k, m_0=m_0)
            fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves=False)
            rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
            val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
            mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
            val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')
            ablation_transfer_housing.loc[len(ablation_transfer_housing)] = [seed, splitting_variable, method, v, source_tree_size, target_tree_size, k, m_0, val_rmse, val_mae, rmse, mae]
            ablation_transfer_housing.to_csv(f'results/LSTransferTreeBoost_ablation_housing.csv')





                            
        



102 2503 2504


KeyboardInterrupt: 

In [19]:
#ablation study for transfertreeboost Gaussian errors, with gaussian source domain errors
ablation_transfer_housing = pd.DataFrame(columns = ['seed', 'splitting_variable', 'method',
                                   'v', 'target_tree_size', 'val_rmse', 'val_mae', 'rmse', 'mae'])

v_list = [0.01, 0.02, 0.05, 0.1, 0.15]
target_tree_size_list = [1,2,3,4]

# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    target_tree_size_list
))



for seed in seed_list:
    for splitting_variable in splitting_variable_list:
    
        #order based on this variable
        data_ = data.sort_values(by=splitting_variable)
        #remove this variable now
        data_ = data_.drop(columns = splitting_variable)
        #divide into source and target data
        data_source = data_[0:int(len(data_) / 4)]
        data_target = data_[int(3*len(data_) / 4):]

        predictor_columns = [col for col in data_.columns if col != target_column]

        #Split target into train, val, test

        data_target_train, data_target_temp = train_test_split(data_target, test_size = 0.98, random_state=seed)
        data_target_val, data_target_test = train_test_split(data_target_temp, test_size = 0.5, random_state=seed)
        print(len(data_target_train), len(data_target_val), len(data_target_test))

        X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

        X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
        X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
        X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])
        for config in param_grid:
            v, target_tree_size = config


            method = 'xgboost'
            params = {
                'objective': 'reg:squarederror',  # Regression with squared error
                'max_depth': target_tree_size,                   # Maximum depth of a tree
                'eta': v,                       # Learning rate
                'eval_metric': 'rmse',           # RMSE as evaluation metric
                }
                    
            bst = train_xgboost(X_target_train, y_target_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
            preds_val = test_xgboost(X_target_val, bst)
            val_rmse = compute_rmse(preds_val, y_target_val)
            val_mae = compute_mae(preds_val, y_target_val)
            preds = test_xgboost(X_target_test, bst)
            rmse = compute_rmse(preds, y_target_test)
            mae = compute_mae(preds, y_target_test)
            ablation_transfer_housing.loc[len(ablation_transfer_housing)] = [seed, splitting_variable, method, v, target_tree_size, val_rmse, val_mae,
                                                                                            rmse,mae]
            
            ablation_transfer_housing.to_csv(f'results/xgboost_ablation_housing.csv')

            method = 'xgboost_naive_transfer'
            params = {
                'objective': 'reg:squarederror',  # Regression with squared error
                'max_depth': target_tree_size,                   # Maximum depth of a tree
                'eta': v,                       # Learning rate
                'eval_metric': 'rmse',           # RMSE as evaluation metric
                }
            X_comb = np.concatenate((X_target_train, X_source_train)) 
            y_comb = np.concatenate((y_target_train, y_source_train))       
            bst = train_xgboost(X_comb, y_comb, X_target_val, y_target_val, boosting_rounds=1000, params=params)
            preds_val = test_xgboost(X_target_val, bst)
            val_rmse = compute_rmse(preds_val, y_target_val)
            val_mae = compute_mae(preds_val, y_target_val)
            preds = test_xgboost(X_target_test, bst)
            rmse = compute_rmse(preds, y_target_test)
            mae = compute_mae(preds, y_target_test)
            ablation_transfer_housing.loc[len(ablation_transfer_housing)] = [seed, splitting_variable, method, v, target_tree_size, val_rmse, val_mae,
                                                                                            rmse,mae]
            
            ablation_transfer_housing.to_csv(f'results/xgboost_ablation_housing.csv')
                                
            



102 2503 2504


In [20]:
#also run mlp finetuning
ablation_transfer_housing = pd.DataFrame(columns = ['seed', 'splitting_variable', 'method', 'base_lr', 'fine_tuning_lr', 'dropout_rate', 'batch_norm',
                                                          'val_rmse', 'val_mae', 'rmse', 'mae'])
fine_tuning_lrs = [1e-4, 5e-5]
base_lrs = [5e-4, 1e-4]
dropout_list = [0.0, 0.1]
include_batch_norm = [True, False]
for seed in seed_list:
    for splitting_variable in splitting_variable_list:
    
        #order based on this variable
        data_ = data.sort_values(by=splitting_variable)
        #remove this variable now
        data_ = data_.drop(columns = splitting_variable)
        #divide into source and target data
        data_source = data_[0:int(len(data_) / 4)]
        data_target = data_[int(3*len(data_) / 4):]

        predictor_columns = [col for col in data_.columns if col != target_column]

        #Split target into train, val, test

        data_target_train, data_target_temp = train_test_split(data_target, test_size = 0.98, random_state=seed)
        data_target_val, data_target_test = train_test_split(data_target_temp, test_size = 0.5, random_state=seed)
        print(len(data_target_train), len(data_target_val), len(data_target_test))

        X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

        X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
        X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
        X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])
        for base_lr in base_lrs:
            for finetuning_lr in fine_tuning_lrs:
                for dropout_rate in dropout_list:
                    for batch_norm in include_batch_norm:

                        method = f'MLP'
                        mlp = MLP(X_target_train.shape[1], 100, 100, 100, 1, dropout_rate=dropout_rate, include_batch_norm=batch_norm)
                        dataloader_train = process_dataset_for_base_network(X_source_train, y_source_train)
                        mlp, train_loss, val_loss = train_mlp_on_source(dataloader_train, mlp, epochs=1000)
                        dataloader_train, dataloader_val, dataloader_test = process_datasets_for_finetuning(X_target_train, y_target_train,
                                                    X_target_val, y_target_val, X_target_test, y_target_test, batch_size=32)
                        
                        mlp, train_loss, val_loss = finetune_mlp_on_target(dataloader_train, dataloader_val, mlp, epochs=1000, freeze_layers=None)
                        val_rmse, val_mae = test_final_mlp(dataloader_val, mlp)
                        rmse, mae = test_final_mlp(dataloader_test, mlp)
                        print(rmse)
                        ablation_transfer_housing.loc[len(ablation_transfer_housing)] = [seed, splitting_variable, method, base_lr, finetuning_lr, dropout_rate, batch_norm, val_rmse, 
                                                                                                        val_mae, rmse, mae]
                        ablation_transfer_housing.to_csv(f'results/MLP_ablation_housing.csv')



102 2503 2504
335347.7
1471368.8
335523.97


KeyboardInterrupt: 